# Round mosaics — live quick look at every round

Builds a mosaic (one per real imaging color) for whichever rounds you select
(`SELECTED_ROUNDS`), from a single frame per FOV near `TARGET_Z_UM` -- no
full z-stack read, so it's light enough to run continuously alongside a real
acquisition (reading straight off the NAS while HAL/Dave is still writing).
Flat-field correction is OFF by default (`ENABLE_FFC`, section 2 -- see
`MERci.live_round_mosaic`'s module docstring for the FFC rationale) for speed.
By default, the 488 nm bead/focus-lock reference channel is excluded from
every round (`EXCLUDED_COLORS`, section 2) -- it's always at a fixed bead z
regardless of `TARGET_Z_UM`, not real tissue signal. This is a quick-look
tool, not a replacement for `after_imaging/02_round_scheduler.ipynb`'s production
mosaics (mid-z, optional FFC, built only once a round is 100% done) -- both
can run at the same time without conflicting; this notebook saves to
`SAMPLE_DIR/figures/`, not `analysis/mosaics/`.

**One state table, one loop, three use cases.** Section 6 builds a small
table -- one row per selected round, with how many of its FOVs are imaged
vs. already have a computed mosaic thumbnail -- and section 7 processes
whatever's outstanding, round by round, in the order the table lists them:
- `SELECTED_ROUNDS = "cells"`-only (or any single round) + `LIVE_LOOP = False`
  -- build it once from whatever's imaged right now, then stop. The old
  "on-demand specific round" mode.
- `SELECTED_ROUNDS = "all"` + `LIVE_LOOP = False` -- one pass over every
  round that's already fully imaged but not yet processed. The old
  "catch-up pass" mode.
- `SELECTED_ROUNDS = "all"` + `LIVE_LOOP = True` (the default) -- keeps
  re-checking and re-processing until every selected round is both fully
  imaged AND fully processed, picking up new rounds as imaging progresses.
  The old "live loop" mode -- meant to be started once and left running for
  the whole experiment.

Mix and match freely -- e.g. `SELECTED_ROUNDS = ["cells", 6]` with
`LIVE_LOOP = True` watches just those two rounds and stops once both are
done, ignoring everything else.

**Sequential only, by design, at least for now.** Parallelizing this across
a SLURM cluster (one array job per FOV, mirroring
`07_cluster_submit_analysis.ipynb`) was considered and deliberately deferred
-- that pattern exists in this repo for a much heavier per-FOV task (a full
multi-frame z-stack read, budgeted at 2 hours per SLURM task); this
notebook's per-FOV cost is a single named frame plus a cheap thumbnail, and
SLURM's own job-submission/queue overhead would plausibly dominate a task
that light rather than speed it up. Revisit this (or a local
`ProcessPoolExecutor`, matching `FOVScheduler`'s own pattern, which avoids
SLURM's overhead entirely) only if a real bulk backlog turns out too slow
sequentially.

**Reading from the NAS while it's being written to**: this notebook's reads
and HAL's writes share the same underlying disk/network link, so there's a
real, if usually small, risk of one slowing the other down -- see
`CATCHUP_READ_DELAY_SEC` (section 2) and
`LiveRoundMosaicBuilder.build_round_mosaic`'s docstring for the rationale
behind its default.

## 1 — Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

from IPython.display import display, clear_output

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.live_round_mosaic       import LiveRoundMosaicBuilder, register_focustest_round
from MERci.plots.round_mosaic_plots import show_round_mosaic
from MERci.visualization import get_merci_figures_dir

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX  = ".zarr"   # must match what HAL is writing
NOTEBOOK_NAME = "round_mosaics"   # used to namespace this notebook's cache + figure files

# Camera-to-stage orientation (MERlin's own transpose/flip_horizontal/
# flip_vertical convention, data/configs/merlin/microscope/*.json) -- applied
# to every raw frame before it becomes a tile, so mosaic content lines up
# with its neighbours instead of appearing rotated/mirrored relative to the
# FOV grid (see section 5). MF2-MF5: 0.108 um/px, 2048 px | MFX/ST2: 0.0878
# um/px, 2304 px.
MICROSCOPE = "MF3"

# Real stage z (um) to build every round's mosaic at -- for each round/color,
# the frame whose own z is closest to this gets used. One frame per FOV per
# color, no z-stack -- deliberately light enough to run continuously during a
# real acquisition. Change and re-run section 3 to pick a different depth
# (e.g. to match where your tissue actually has signal).
TARGET_Z_UM = 10.0

# Colors to leave out of every round's resolution (section 3) -- by default
# just the 488 nm bead/reference channel, imaged only for HAL's own focus
# lock at a fixed bead z regardless of TARGET_Z_UM (not real tissue signal,
# and always far from TARGET_Z_UM -- the actual source of a ">5um away"
# warning otherwise firing on nearly every round). Set to [] to include
# every real color again, e.g. to build a bead-channel mosaic for a specific
# round (combine with SELECTED_ROUNDS below rather than watching/catching
# up on beads for every round).
EXCLUDED_COLORS = [488.0]

# Which round(s) to build/watch mosaics for (section 6/7) -- "all" (default)
# considers every round with at least one resolved color; otherwise a list
# mixing round labels ("cells", resolved by imaging_type -- same convention
# as correct_camera_rotation.ipynb's own ROUND_IMAGING_TYPE) and/or explicit
# imaging_round numbers, e.g. ["cells", 6]. Processed in the order given
# (or ascending round id, for "all").
SELECTED_ROUNDS = "all"

# True (default): keep re-checking/re-processing until every selected round
# is both fully imaged AND fully processed (picking up newly-imaged FOVs and
# newly-started rounds as they appear) -- meant to be started once and left
# running for the whole experiment. False: process whatever's available
# right now, once, then stop -- e.g. for a quick one-off look at a single
# round (SELECTED_ROUNDS="cells", LIVE_LOOP=False), or one catch-up pass
# over everything already finished (SELECTED_ROUNDS="all", LIVE_LOOP=False).
LIVE_LOOP = True

# Flat-field correction -- off by default (fastest option, no extra reads).
# Turning it on costs FFC_N_FOVS extra one-time reads PER COLOR (not per
# round -- vignetting is a fixed optical property, cached to disk after the
# first computation) -- see section 5.
ENABLE_FFC = False
FFC_N_FOVS = 10

# Shared contrast range (vmin/vmax) for every tile in a round/color, estimated
# once from the pooled pixel histogram of CONTRAST_SAMPLE_N_FOVS random
# INTERIOR (non-exterior) FOVs -- interior FOVs are more likely to carry real
# tissue signal than exterior/border ones (which FFC's own sampling already
# prefers for the opposite reason: likely near-empty). Keeps every tile
# visually consistent without needing the whole round done first, unlike a
# single shared whole-canvas stretch computed only after the fact -- see
# section 5.
CONTRAST_SAMPLE_N_FOVS = 10
CONTRAST_LOW_PCT  = 5.0
CONTRAST_HIGH_PCT = 99.0

POLL_INTERVAL_SEC = 5         # must be well under the time to acquire one FOV
MAX_RUNTIME_MIN    = 24*60*7  # safety cap -- this notebook is meant to run for a
                               # whole multi-round experiment, not just one round

# How often (seconds) the on-screen mosaic figure actually redraws while a
# round is being built tile by tile (section 5/7) -- every tile is placed
# into its canvas immediately regardless, this only paces how often the
# figure itself is re-rendered/re-displayed, so many cheap tile placements
# (e.g. loading already-cached thumbnails) don't spend more wall-clock time
# drawing matplotlib figures than actually reading data.
LIVE_REDRAW_MIN_INTERVAL_SEC = 0.5

# Max long-side pixel size for the ON-SCREEN live preview only (section 5's
# show_round_mosaic) -- NOT the full-resolution PNG saved to figures/, which
# is unaffected. At real production scale (LT060_sample_04: 1166 FOVs, a
# ~10600x6700 px canvas) matplotlib's own imshow+draw of the FULL-resolution
# array measured at ~6s, confirmed directly -- since maybe_redraw() above
# fires roughly every LIVE_REDRAW_MIN_INTERVAL_SEC, that alone would dominate
# wall-clock time and make "live" updates arrive every several seconds at
# best regardless of how fast the underlying FOV reads are, which is exactly
# what defeated the point of per-tile live redraws once tested at real scale
# (the fix that introduced them was only ever verified against 20 small fake
# FOVs). Downsampling the array BEFORE imshow drops this to well under 0.1s,
# confirmed directly, independent of the real canvas size.
LIVE_PREVIEW_MAX_PX = 1400

# How often (seconds) the FULL-RESOLUTION mosaic PNG on disk actually gets
# rewritten while a round builds live -- deliberately much coarser than
# LIVE_REDRAW_MIN_INTERVAL_SEC, since writing the real full-resolution PNG
# (not the downsampled preview above) is itself the expensive part: ~3s for
# LT060_sample_04's real canvas, confirmed directly. The on-screen preview
# stays live every LIVE_REDRAW_MIN_INTERVAL_SEC regardless of this setting;
# this only bounds how stale the saved PNG FILE can get during a long
# catch-up pass (it's always refreshed at least once more per
# build_round_mosaic call regardless, via the unconditional final
# maybe_redraw(force=True)).
DISK_SAVE_MIN_INTERVAL_SEC = 20.0

# Pacing gap between successive fresh (non-cached) FOV reads during a
# catch-up burst -- see MERci.live_round_mosaic's module docstring and
# LiveRoundMosaicBuilder.build_round_mosaic for the rationale (shared
# disk/network link with HAL's own writes).
CATCHUP_READ_DELAY_SEC = 0.02

config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

# Focus-lock test (before_imaging's create_focus_test_dave_config) is a
# standalone calibration procedure, not a real round in round_info.csv --
# register_focustest_round adds it as a synthetic round_id=0 (see its own
# docstring) whenever this experiment actually has a focus-test HAL config,
# so it shows up in section 3/6/7 exactly like any other round.
FOCUSTEST_ROUND_ID = 0
INCLUDE_FOCUSTEST_ROUND = True   # False to skip registering it even if present

if INCLUDE_FOCUSTEST_ROUND:
    if register_focustest_round(meta, config, MICROSCOPE, FOCUSTEST_ROUND_ID):
        print(f"Registered focus-test round (round_id={FOCUSTEST_ROUND_ID}).")
    else:
        print(f"No focus-test HAL config found for {MICROSCOPE} (or round "
              f"{FOCUSTEST_ROUND_ID} already registered) -- skipping.")

THUMBNAILS_DIR = config.analysis_dir / "thumbnails"   # shared with 01_fov_scheduler.ipynb's own convention
THUMBNAILS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = config.analysis_dir / "cache" / NOTEBOOK_NAME   # FFC fields only (NOTEBOOK_GUIDELINES.md #2)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
# Deliberately SAMPLE_DIR/figures/, not analysis/mosaics/ -- see markdown
# above: this is a quick-look tool, kept out of the way of the production
# mosaics after_imaging/02_round_scheduler.ipynb builds at the same filenames.
FIGURES_DIR = get_merci_figures_dir(SAMPLE_DIR, "during_imaging", NOTEBOOK_NAME)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample name  : {SAMPLE_NAME}")
print(f"MICROSCOPE   : {MICROSCOPE}")
print(f"FOVs (total) : {meta.n_fovs}")
print(f"Rounds       : {sorted(meta.rounds)}")
print(f"TARGET_Z_UM  : {TARGET_Z_UM}")
print(f"ENABLE_FFC   : {ENABLE_FFC}")
print(f"LIVE_LOOP    : {LIVE_LOOP}")

MICROSCOPE_ORIENTATION_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "microscope"
MOSAIC_BUILDER = LiveRoundMosaicBuilder(
    config=config, metadata=meta,
    microscope=MICROSCOPE, microscope_orientation_dir=MICROSCOPE_ORIENTATION_DIR,
    cache_dir=CACHE_DIR, figures_dir=FIGURES_DIR, thumbnails_dir=THUMBNAILS_DIR,
    notebook_name=NOTEBOOK_NAME,
    target_z_um=TARGET_Z_UM, excluded_colors=EXCLUDED_COLORS,
    focustest_round_id=FOCUSTEST_ROUND_ID,
    enable_ffc=ENABLE_FFC, ffc_n_fovs=FFC_N_FOVS,
    contrast_sample_n_fovs=CONTRAST_SAMPLE_N_FOVS,
    contrast_low_pct=CONTRAST_LOW_PCT, contrast_high_pct=CONTRAST_HIGH_PCT,
    live_preview_max_px=LIVE_PREVIEW_MAX_PX,
    live_redraw_min_interval_sec=LIVE_REDRAW_MIN_INTERVAL_SEC,
    disk_save_min_interval_sec=DISK_SAVE_MIN_INTERVAL_SEC,
    catchup_read_delay_sec=CATCHUP_READ_DELAY_SEC,
)

## 3 — Resolve each round's real colors -> nearest-z frame index

Reads each round's own frame table (via its HAL config, same resolution
`before_imaging` already uses) and picks, per real color (excluding
`EXCLUDED_COLORS`, section 2), the frame whose actual `z` (um) is closest to
`TARGET_Z_UM`. Rounds sharing the same HAL config (common for repeated bits
rounds) resolve identically, but each round is still looked up independently
since nothing guarantees that in general.

In [ ]:
ROUND_COLOR_FRAMES = {}
for round_id in sorted(meta.rounds):
    cf = MOSAIC_BUILDER.resolve_round_color_frames(round_id)
    if cf:
        ROUND_COLOR_FRAMES[round_id] = cf
    print(f"Round {round_id}: colors {sorted(cf)} -> frame indices {cf}")

## 4 — Resolve round labels and `SELECTED_ROUNDS`

A round's label is `"cells"` if one of its series has `imaging_type=="cells"`,
else its raw `imaging_round` number -- matching how `SELECTED_ROUNDS`
entries are themselves resolved (a string by `imaging_type`, an int
directly).

In [ ]:
if SELECTED_ROUNDS == "all":
    SELECTED_ROUND_IDS = sorted(ROUND_COLOR_FRAMES)
else:
    # A bare string ("cells") means one round, not a list of its characters --
    # iterating a str yields its letters one at a time, which previously sent
    # single characters like "c" into resolve_round_token and raised a
    # confusing ValueError. Wrap it into a one-element list instead.
    selected_tokens = [SELECTED_ROUNDS] if isinstance(SELECTED_ROUNDS, str) else SELECTED_ROUNDS
    SELECTED_ROUND_IDS = [MOSAIC_BUILDER.resolve_round_token(t) for t in selected_tokens]
    unresolved = [r for r in SELECTED_ROUND_IDS if r not in ROUND_COLOR_FRAMES]
    if unresolved:
        raise ValueError(f"Round id(s) {unresolved} have no resolved colors "
                          f"(see section 3) -- check round_info.csv.")

print(f"SELECTED_ROUND_IDS: {SELECTED_ROUND_IDS} "
      f"(labels: {[MOSAIC_BUILDER.round_label_for(r) for r in SELECTED_ROUND_IDS]})")

## 5 — Mosaic builder stats

`MOSAIC_BUILDER` (constructed in section 2 -- `MERci.live_round_mosaic.LiveRoundMosaicBuilder`)
holds this session's FFC fields, contrast ranges, tile-placement geometry,
and in-progress canvases across repeated polls, and does the actual tile
reading/correcting/placing in section 6/7 below. See that module's own
docstring (and `LiveRoundMosaicBuilder.build_round_mosaic`'s) for the full
tile-by-tile / live-redraw / camera-orientation / FFC / contrast-range
design and rationale -- kept out of this notebook so it lives in one place
next to the code it explains.

In [ ]:
print(f"Microscope orientation ({MICROSCOPE}): {MOSAIC_BUILDER.microscope_orientation}")
print(f"STEP_SIZE_UM       : {MOSAIC_BUILDER.step_size_um:.1f} um")
print(f"Exterior FOV count : {len(MOSAIC_BUILDER.exterior_fov_ids)}")

## 6 — Build the state table

One row per selected round: how many of its FOVs are imaged right now, how
many already have a computed thumbnail for every one of that round's colors
("processed"), and the experiment's total FOV count. Re-run this cell any
time for a fresh snapshot -- section 7 also rebuilds it itself on every
poll, so this is mainly for a quick look without starting the full loop.

In [ ]:
STATE_DF = MOSAIC_BUILDER.build_state_df(SELECTED_ROUND_IDS, ROUND_COLOR_FRAMES)
display(STATE_DF)

## 7 — Process selected rounds (one loop for all three use cases)

Rebuilds the state table every cycle and processes any round with
`processed_fovs < imaged_fovs`, in the order `SELECTED_ROUND_IDS` lists them
-- reading/thumbnailing just the FOVs that are imaged but not yet processed.
The mosaic figure updates FOV by FOV as each tile is read
(`LiveRoundMosaicBuilder.build_round_mosaic`/`show_round_mosaic`, throttled
to `LIVE_REDRAW_MIN_INTERVAL_SEC`), not only once a round's whole batch of
pending FOVs finishes. If `LIVE_LOOP`, keeps looping until every selected
round is both fully IMAGED and fully PROCESSED (or `MAX_RUNTIME_MIN` is
exceeded); otherwise processes whatever's available once and stops. Interrupt
the kernel to stop early at any time -- whatever's been built so far is
already saved to `figures/`.

In [ ]:
start_time = time.time()
poll_count = 0
last_processed_round_id = None   # most recent round build_round_mosaic actually built --
                                  # see the idle-cycle handling below

try:
    while True:
        if (time.time() - start_time) > MAX_RUNTIME_MIN * 60:
            print(f"Stopping: MAX_RUNTIME_MIN={MAX_RUNTIME_MIN} exceeded.")
            break

        poll_count += 1
        state_df = MOSAIC_BUILDER.build_state_df(SELECTED_ROUND_IDS, ROUND_COLOR_FRAMES)
        fully_done = True
        processed_anything = False

        for _, row in state_df.iterrows():
            round_id = row["round_id"]
            if row["imaged_fovs"] < row["total_fovs"]:
                fully_done = False   # still being imaged -- keep watching even with nothing to process yet
            if row["processed_fovs"] < row["imaged_fovs"]:
                fully_done = False
                processed_anything = True
                color_frames = ROUND_COLOR_FRAMES[round_id]
                fov_ids = MOSAIC_BUILDER.round_imaged_fov_ids(round_id)
                print(f"poll #{poll_count} | round {row['round']}: processing "
                      f"{row['processed_fovs']} -> {len(fov_ids)}/{row['total_fovs']} FOVs...")
                MOSAIC_BUILDER.build_round_mosaic(round_id, color_frames, fov_ids, row["round"])
                last_processed_round_id = round_id

        # Refresh before printing -- state_df above was captured at the TOP
        # of this cycle, so it would otherwise still show pre-processing
        # counts for whatever this cycle just built, right below the mosaic
        # image proving it was actually built.
        state_df = MOSAIC_BUILDER.build_state_df(SELECTED_ROUND_IDS, ROUND_COLOR_FRAMES)
        elapsed = time.time() - start_time

        # Idle cycles (nothing to process -- every round already at
        # processed_fovs == imaged_fovs, e.g. waiting between rounds, or an
        # aborted acquisition that will never reach total_fovs) never call
        # build_round_mosaic, so nothing ever clears the output area on its
        # own: left alone, this poll's status print just appends below every
        # previous idle poll's, growing without bound (confirmed directly:
        # 130+ stacked tables in one real run). A bare clear_output() fixes
        # the spam but overcorrects -- it wipes the WHOLE cell output,
        # including the mosaic figure a prior active cycle displayed, so an
        # idle stretch blanked the in-progress mosaic out of view too,
        # leaving only this status table (confirmed directly: a real
        # aborted run kept idle-polling with the mosaic gone from the
        # display). Redraw the last-built canvas instead whenever one
        # exists -- costs nothing new (no fresh I/O, save_full_res=False
        # skips rewriting the PNG) and keeps the image on screen across
        # idle polls, same as an active cycle would.
        if not processed_anything:
            if last_processed_round_id is not None:
                last_color_frames = ROUND_COLOR_FRAMES[last_processed_round_id]
                show_round_mosaic(
                    last_processed_round_id,
                    {c: MOSAIC_BUILDER.get_canvas(last_processed_round_id, c) for c in last_color_frames},
                    MOSAIC_BUILDER.round_label_for(last_processed_round_id),
                    mosaic_paths={c: MOSAIC_BUILDER.mosaic_path(last_processed_round_id, c)
                                  for c in last_color_frames},
                    live_preview_max_px=LIVE_PREVIEW_MAX_PX, save_full_res=False,
                )
            else:
                clear_output(wait=True)
        print(f"poll #{poll_count} | elapsed {elapsed / 60:.1f} min | next check in {POLL_INTERVAL_SEC}s")
        print(state_df.to_string(index=False))

        if not LIVE_LOOP:
            print("LIVE_LOOP is False -- one pass complete.")
            break
        if fully_done:
            print("All selected rounds fully imaged and processed.")
            break

        time.sleep(POLL_INTERVAL_SEC)
except KeyboardInterrupt:
    print(f"Stopped by user after poll #{poll_count}.")